# ICI Treatment Regimens by Mechanism and Response (Supplementary Table S5)

Produces **Table S5**: for the same ICI-treated cohort as Table 1
(`11_demographic_table.ipynb`), breaks Table 1's coarse "ICI Mechanism" row
(Anti-PD1 / Anti-PDL1 / Anti-CTLA4 / Multiple mechanism) down into the
specific drug regimens within each mechanism, each with its own
Responder / Non-Responder counts.

**Row structure:** ICI Mechanism is the row-group header (Anti-PD1,
Anti-PDL1, Anti-CTLA4, Anti-CTLA4/Anti-PD1 [Ipilimumab + Nivolumab, broken
out on its own in Table 1 already], and Multiple mechanism), matching
Table 1's `mechanism` categories exactly so totals agree between the two
tables. Within each mechanism, an **"All regimens"** subtotal row comes
first, followed by that mechanism's individual drug regimens as sub-rows,
ordered by descending N. Raw `treatment` strings are canonicalized via
`normalize_ici_treatment()` (`helpers/shortcuts.r`, using the
`ICI_TREATMENT_NORM` lookup in `helpers/config.r`) so duplicate/reordered
variants of the same regimen (e.g. both "Nivolumab/Ipilimumab" and
"Ipilimumab/Nivolumab") collapse into one row. This applies uniformly to
every mechanism, including **Multiple mechanism**: rather than one vague
row, each specific combination actually observed (e.g. "Ipilimumab +
Pembrolizumab") gets its own sub-row. A final **"All mechanisms"** row
gives the grand total.

Columns are **All treated patients** (a plain count -- everyone in that
row, regardless of response), then **Responder** and **Non-Responder**
(each `n (row %)`: within each regimen's row, the percentage of patients
who were Responders vs Non-Responders — i.e. that regimen's observed
response rate in this cohort). "All regimens" and "All mechanisms" are
subtotal/grand-total rows and should not be summed with the detail rows
beneath them.

Patients with missing `mechanism` or `treatment` are excluded (both are
expected to be populated for every ICI-treated patient, so this should
rarely if ever trigger); the excluded count is reported in a footnote.

**Input:** `cpi_ready.rds`
**Output:**
`figs/table_s5_treatment_by_mechanism_response.csv` -- kept purely as the
intermediate `assemble_supplementary_tables_excel.R` reads to build the combined
`supplementary_tables.xlsx`; no standalone `.html`/`.docx` are generated for this
table (see the combined workbook instead -- note the ICI Mechanism row-group
header there is a plain repeated column, since neither a flat CSV nor an Excel
cell can span rows the way gt/flextable did).

In [ ]:
source(paste0(dirname(getwd()), "/helpers/map.r"))
source(paste0(dirname(getwd()), "/helpers/shortcuts.r"))  # tidyverse + fct_pin_last() + normalize_ici_treatment()
source(paste0(dirname(getwd()), "/helpers/config.r"))

dir.create(paste0(CIDER_PROJ_DIR, "figs/"), showWarnings = FALSE, recursive = TRUE)

In [ ]:
# ── Load ICI-ready data ───────────────────────────────────────────────────────
# Same population as Table 1 and Tables S2, S4: ICI-treated patients with known
# RECIST response, one row per patient.
#
# IMPORTANT: cpi_ready.rds stores one row per (sampleId, gp) *and* a
# duplicated copy of every row with cohort_broad overwritten to
# "ICI-Treated" (added in 4_base.ipynb so Figure 3 can facet pooled-vs-per-
# cancer-type together). The gp filter below removes the first
# duplication; distinct() removes the second.
df_raw <- readRDS(paste0(CIDER_PROJ_DIR, "cpi_ready.rds")) |>
  filter(gp == "T-cell", !is.na(response)) |>
  distinct(sampleId, .keep_all = TRUE)

In [ ]:
# ── Variable preparation ──────────────────────────────────────────────────────
# ICI Mechanism: identical field and pinned ordering to Table 1
# (11_demographic_table.ipynb), so this table's mechanism-level totals
# match Table 1's "ICI Mechanism" row exactly.
#
# Treatment: raw `treatment` regimen strings, canonicalized via
# normalize_ici_treatment() (helpers/shortcuts.r) so duplicate/reordered
# variants of the same regimen collapse into one row. Falls back to the
# raw string for anything the hand-curated lookup doesn't cover, rather
# than dropping it.
demo <- df_raw |>
  mutate(
    `ICI Mechanism` = fct_pin_last(factor(mechanism), "Multiple mechanism"),
    Treatment        = normalize_ici_treatment(treatment),
    Responder        = factor(responder, levels = c("Responder", "Non-Responder"))
  )

known <- demo |> filter(!is.na(`ICI Mechanism`), !is.na(Treatment))

n_total    <- nrow(demo)
n_known    <- nrow(known)
n_excluded <- n_total - n_known
if (n_excluded > 0) {
  message(sprintf(
    "Note: %d of %d ICI-treated patients excluded from Table S5 (missing mechanism: %d, missing treatment: %d).",
    n_excluded, n_total,
    sum(is.na(demo$`ICI Mechanism`)),
    sum(is.na(demo$Treatment))
  ))
}

mechanism_levels <- levels(demo$`ICI Mechanism`)
resp_order        <- c("All treated patients", "Responder", "Non-Responder")

In [ ]:
# ── Build nested rows: Mechanism -> [All regimens, then each Treatment] ───────
# One mechanism at a time (in mechanism_levels order -- descending count,
# "Multiple mechanism" pinned last), build an "All regimens" subtotal row
# followed by that mechanism's individual regimens ordered by descending N.
# purrr::map_dfr() assembles these in exactly that order; nothing below is
# re-sorted afterwards, since tidyr::pivot_wider() preserves the row order
# of first appearance of its id columns.
summarise_responses <- function(df) {
  df |>
    count(Responder) |>
    mutate(label = sprintf("%d (%d%%)", n, round(100 * n / sum(n)))) |>
    select(Responder, label) |>
    bind_rows(tibble(Responder = "All treated patients", label = as.character(nrow(df))))
}

build_mechanism_block <- function(df, mech) {
  df_m <- df |> filter(`ICI Mechanism` == mech)

  subtotal <- summarise_responses(df_m) |> mutate(Treatment = "All regimens")

  treatments_in_order <- df_m |> count(Treatment, sort = TRUE) |> pull(Treatment)
  detail <- purrr::map_dfr(treatments_in_order, function(tx) {
    summarise_responses(df_m |> filter(Treatment == tx)) |> mutate(Treatment = tx)
  })

  bind_rows(subtotal, detail) |> mutate(`ICI Mechanism` = mech)
}

mechanism_blocks <- purrr::map_dfr(mechanism_levels, ~ build_mechanism_block(known, .x))

grand_total <- summarise_responses(known) |>
  mutate(Treatment = "All regimens", `ICI Mechanism` = "All mechanisms")

all_rows <- bind_rows(mechanism_blocks, grand_total)

wide <- all_rows |>
  mutate(Responder = factor(Responder, levels = resp_order)) |>
  tidyr::pivot_wider(names_from = Responder, values_from = label, values_fill = "\u2014")

# NOTE: intersect() must check names(wide) (the just-pivoted table), not
# names(all_rows) (the pre-pivot long table) -- the latter's columns are
# Responder/label/Treatment/ICI Mechanism, so intersecting resp_order
# against it only ever matches "Responder" (a coincidental name collision)
# and silently drops "All treated patients"/"Non-Responder" from the output.
wide <- wide |>
  select(`ICI Mechanism`, Treatment, all_of(intersect(resp_order, names(wide))))

wide

In [ ]:
# ── Save outputs ──────────────────────────────────────────────────────────────
out_dir <- paste0(CIDER_PROJ_DIR, "figs/")

footnote_text <- if (n_excluded > 0) {
  sprintf(
    paste0(
      "%d of %d ICI-treated patients (Table 1's cohort) are excluded here due to ",
      "missing ICI Mechanism or Treatment data. \"All treated patients\" is a plain ",
      "count; Responder/Non-Responder cells report n (%% of that regimen's ",
      "patients). \"All regimens\" and \"All mechanisms\" are subtotal/grand-total ",
      "rows and should not be summed with the detail rows beneath them."
    ),
    n_excluded, n_total
  )
} else {
  sprintf(
    paste0(
      "All %d ICI-treated patients (Table 1's cohort) are included. \"All treated ",
      "patients\" is a plain count; Responder/Non-Responder cells report n (%% of ",
      "that regimen's patients). \"All regimens\" and \"All mechanisms\" are ",
      "subtotal/grand-total rows and should not be summed with the detail rows ",
      "beneath them."
    ),
    n_total
  )
}

# CSV — flat table (both ICI Mechanism and Treatment as ordinary columns,
# since CSV has no notion of a row-group header). Build intermediate only
# -- not a standalone deliverable: this table's actual output is its sheet
# in the combined supplementary_tables.xlsx
# (assemble_supplementary_tables_excel.R reads this .csv + the metadata
# sidecar below to build it).
wide |>
  readr::write_csv(paste0(out_dir, "table_s5_treatment_by_mechanism_response.csv"))

message("Saved: table_s5_treatment_by_mechanism_response.csv")

# Metadata sidecar (title + footnote) -- consumed by
# assemble_supplementary_tables_excel.R to build the combined supplemental
# tables Excel workbook without re-deriving this table's title/footnote
# text (footnote_text is computed above).
jsonlite::write_json(
  list(
    title    = "Supplementary Table S5. ICI treatment regimens by mechanism and response",
    footnote = footnote_text
  ),
  paste0(out_dir, "table_s5_treatment_by_mechanism_response_meta.json"),
  auto_unbox = TRUE
)